# StutterSense: Annotator-Aware Stuttering Event Detection

**Research question:** Does preserving annotator disagreement improve stuttering-event classification, calibration, and uncertainty estimation relative to majority-vote labels?

## Design

Five event types are modeled independently from frozen WavLM Base+ embeddings:

1. **Hard BCE:** majority targets, `count >= 2`.
2. **Soft BCE:** annotator proportions, `count / 3`.
3. **Soft BCE + Brier:** soft targets with a calibration penalty.

Counts of 1 or 2 indicate aggregate ambiguity; individual rater identities are unavailable. Evaluation is duplicate-aware but may contain speaker leakage because the hosted schema exposes no verified speaker or episode IDs. Findings are exploratory and not clinical predictions.

## 1. Environment

Install `requirements.txt` before running the notebook. The data and model are public; no API key or local dataset is required.

In [ ]:
import importlib.util
import sys

REQUIRED_IMPORTS = {
    "datasets": "datasets[audio]",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "soundfile": "soundfile",
    "librosa": "librosa",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "ipywidgets": "ipywidgets",
    "huggingface_hub": "huggingface-hub",
    "torch": "torch",
}
missing_packages = [package for module, package in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing dependencies: " + ", ".join(missing_packages)
        + ". From the repository root, run: python -m pip install -r requirements.txt"
    )
print("Dependency check passed for", sys.executable)

In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import random
import shutil
from datetime import datetime, timezone
from pathlib import Path

import datasets
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn
import soundfile as sf
import torch
import transformers
from datasets import Audio, load_dataset
from IPython.display import Audio as IPythonAudio, FileLink, Markdown, clear_output, display
from huggingface_hub import HfApi
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, WavLMModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | device: {DEVICE}")
print(f"transformers: {transformers.__version__} | datasets: {datasets.__version__}")
print(f"numpy: {np.__version__} | scipy: {scipy.__version__} | sklearn: {sklearn.__version__}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

Defaults target a GPU with approximately 16 GB of memory. Use `FAST_SMOKE_TEST` or `CPU_SMOKE_TEST` for reduced runs. `MAX_SAMPLES=None` materializes the full hosted split. Set `STUTTERSENSE_OUTPUT_DIR` to redirect outputs.

In [ ]:
DATASET_ID = "psusac/sep28k"
DATASET_CONFIG = "default"
DATASET_REVISION = None  # Set a Hugging Face commit hash to pin the source explicitly.
MAX_SAMPLES = 2000
BATCH_SIZE = 8
EPOCHS = 12
LEARNING_RATE = 1e-3
RANDOM_SEED = 42
SAMPLE_RATE = 16000
MODEL_NAME = "microsoft/wavlm-base-plus"
OUTPUT_DIR = os.environ.get("STUTTERSENSE_OUTPUT_DIR", "outputs")

FAST_SMOKE_TEST = False
CPU_SMOKE_TEST = False
SHUFFLE_BUFFER = 10_000
EARLY_STOPPING_PATIENCE = 3
DROPOUT = 0.20
BRIER_LAMBDA = 0.25
USE_DISAGREEMENT_WEIGHTS = False
DISAGREEMENT_WEIGHT_STRENGTH = 0.5
TUNE_THRESHOLDS_ON_VALIDATION = False
SHOW_OUTPUT_ZIP_LINK = True

EVENTS = ["prolongation", "block", "sound_rep", "word_rep", "interjection"]
DIAGNOSTIC_COLUMNS = [
    "unsure", "poor_audio_quality", "no_speech", "difficult_to_understand",
    "no_stuttered_words", "natural_pause", "music",
]

RUN_MAX_SAMPLES = MAX_SAMPLES
RUN_BATCH_SIZE = BATCH_SIZE
RUN_EPOCHS = EPOCHS
if FAST_SMOKE_TEST:
    RUN_MAX_SAMPLES = 64 if MAX_SAMPLES is None else min(MAX_SAMPLES, 64)
    RUN_BATCH_SIZE = min(BATCH_SIZE, 4)
    RUN_EPOCHS = min(EPOCHS, 2)
if CPU_SMOKE_TEST:
    DEVICE = torch.device("cpu")
    RUN_MAX_SAMPLES = 32 if RUN_MAX_SAMPLES is None else min(RUN_MAX_SAMPLES, 32)
    RUN_BATCH_SIZE = min(RUN_BATCH_SIZE, 2)
    RUN_EPOCHS = 1

OUTPUT_PATH = Path(OUTPUT_DIR)
FIGURE_PATH = OUTPUT_PATH / "figures"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

def set_seed(seed=RANDOM_SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
sns.set_theme(style="whitegrid", context="notebook")
print({"seed": RANDOM_SEED, "device": str(DEVICE), "samples": RUN_MAX_SAMPLES,
       "batch_size": RUN_BATCH_SIZE, "epochs": RUN_EPOCHS,
       "output_dir": str(OUTPUT_PATH.resolve())})

## 3. Data loading and validation

The hosted schema is validated at runtime. Audio is decoded from bytes or paths, converted to mono, and resampled to 16 kHz without truncation.

In [ ]:
dataset_hub_info = HfApi().dataset_info(DATASET_ID, revision=DATASET_REVISION)
RESOLVED_DATASET_REVISION = dataset_hub_info.sha
print("Resolved dataset revision:", RESOLVED_DATASET_REVISION)

raw_stream = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    streaming=True,
    revision=RESOLVED_DATASET_REVISION,
)
print("Hosted schema:")
print(raw_stream.features)
available_columns = list(raw_stream.features.keys())
required_columns = ["audio", *EVENTS]
missing_columns = [c for c in required_columns if c not in available_columns]
if missing_columns:
    raise RuntimeError(f"Hosted schema is missing required columns: {missing_columns}")

identifier_candidates = [
    c for c in available_columns
    if any(token in c.lower() for token in ("speaker", "episode", "clip_id", "file", "path", "id"))
    and c != "audio"
]
print("Verified identifier-like columns:", identifier_candidates or "none")
print("Diagnostic columns present:", [c for c in DIAGNOSTIC_COLUMNS if c in available_columns])

raw_stream = raw_stream.cast_column("audio", Audio(decode=False))
raw_stream = raw_stream.shuffle(seed=RANDOM_SEED, buffer_size=SHUFFLE_BUFFER)

In [ ]:
def decode_audio_value(value):
    '''Return mono float32 samples and source rate from common HF audio representations.'''
    if hasattr(value, "get_all_samples"):
        decoded = value.get_all_samples()
        array = decoded.data.detach().cpu().numpy() if torch.is_tensor(decoded.data) else np.asarray(decoded.data)
        source_rate = int(decoded.sample_rate)
    elif isinstance(value, dict) and value.get("array") is not None:
        array = np.asarray(value["array"])
        source_rate = int(value["sampling_rate"])
    else:
        payload = None
        if isinstance(value, dict):
            if value.get("bytes") is not None:
                payload = io.BytesIO(value["bytes"])
            elif value.get("path"):
                payload = value["path"]
        elif isinstance(value, (bytes, bytearray)):
            payload = io.BytesIO(value)
        elif isinstance(value, (str, os.PathLike)):
            payload = str(value)
        if payload is None:
            raise TypeError(f"Unsupported audio value type: {type(value)!r}")
        try:
            array, source_rate = sf.read(payload, dtype="float32", always_2d=False)
        except Exception:
            if hasattr(payload, "seek"):
                payload.seek(0)
            array, source_rate = librosa.load(payload, sr=None, mono=False)

    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        channel_axis = 0 if array.shape[0] <= array.shape[1] else 1
        array = array.mean(axis=channel_axis)
    if array.ndim != 1 or array.size == 0:
        raise ValueError(f"Invalid decoded waveform shape: {array.shape}")
    if not np.isfinite(array).all():
        raise ValueError("Waveform contains NaN or infinity")
    source_rate = int(source_rate)
    if source_rate <= 0:
        raise ValueError(f"Invalid sample rate: {source_rate}")
    if source_rate != SAMPLE_RATE:
        array = librosa.resample(array, orig_sr=source_rate, target_sr=SAMPLE_RATE)
    return np.ascontiguousarray(array, dtype=np.float32), source_rate


records = []
waveforms = []
decode_failures = []
attempted = 0

for example in tqdm(raw_stream, desc="Streaming and decoding clips"):
    if RUN_MAX_SAMPLES is not None and len(records) >= RUN_MAX_SAMPLES:
        break
    attempted += 1
    try:
        counts = np.asarray([example[name] for name in EVENTS], dtype=np.float32)
        if not np.isfinite(counts).all() or np.any(counts < 0) or np.any(counts > 3):
            raise ValueError(f"Annotation counts outside [0, 3]: {counts.tolist()}")
        if not np.allclose(counts, np.round(counts)):
            raise ValueError(f"Non-integer annotation counts: {counts.tolist()}")
        waveform, original_rate = decode_audio_value(example["audio"])
        fingerprint = hashlib.sha256(waveform.tobytes()).hexdigest()
        row = {name: int(example[name]) for name in EVENTS}
        for name in DIAGNOSTIC_COLUMNS:
            if name in example and example[name] is not None:
                row[name] = int(example[name])
        for name in identifier_candidates:
            if example.get(name) is not None:
                row[name] = str(example[name])
        row.update({
            "source_index": attempted - 1,
            "original_sample_rate": original_rate,
            "num_samples": int(waveform.size),
            "duration_seconds": float(waveform.size / SAMPLE_RATE),
            "audio_sha256": fingerprint,
        })
        records.append(row)
        waveforms.append(waveform)
    except Exception as exc:
        decode_failures.append({"source_index": attempted - 1, "error": repr(exc)})

if not records:
    raise RuntimeError("No clips decoded successfully; inspect the failure log above.")

metadata = pd.DataFrame(records)
counts_matrix = metadata[EVENTS].to_numpy(dtype=np.float32)
hard_targets = (counts_matrix >= 2).astype(np.float32)
soft_targets = counts_matrix / 3.0
durations = metadata["duration_seconds"].to_numpy()
unusual_mask = (durations < 0.25) | (durations > 30.0)

print(f"Decoded {len(records):,} clips after attempting {attempted:,}; failures: {len(decode_failures):,}")
if decode_failures:
    display(pd.DataFrame(decode_failures).head(20))
print(metadata["duration_seconds"].describe())
print(f"Unusual-duration clips (<0.25 s or >30 s): {int(unusual_mask.sum())}")
if unusual_mask.any():
    display(metadata.loc[unusual_mask, ["source_index", "duration_seconds", "audio_sha256"]].head(20))
print("Duplicate waveform fingerprints:", int(metadata["audio_sha256"].duplicated().sum()))
display(metadata.head())

In [ ]:
print("A few streamed examples (playback is optional):")
for i in range(min(3, len(waveforms))):
    display(Markdown(f"**Clip {i}** — {metadata.loc[i, EVENTS].to_dict()}"))
    display(IPythonAudio(waveforms[i], rate=SAMPLE_RATE))

## 4. Exploratory analysis

Counts of 1 or 2 define the disagreement proxy. This reflects aggregate ambiguity, not rater-specific behavior. Co-occurrence uses majority-vote labels.

In [ ]:
def save_current_figure(filename):
    path = FIGURE_PATH / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved", path)

prevalence = pd.Series(hard_targets.mean(axis=0), index=EVENTS)
ax = prevalence.sort_values().plot(kind="barh", color="#3B82F6", figsize=(8, 4.5))
ax.set(xlabel="Majority-vote prevalence", ylabel="Event", title="A. Dataset label frequency")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
save_current_figure("A_dataset_label_frequency.png")
plt.show()

count_long = metadata[EVENTS].melt(var_name="event", value_name="annotation_count")
count_table = pd.crosstab(count_long["event"], count_long["annotation_count"]).reindex(columns=[0, 1, 2, 3], fill_value=0)
count_table.plot(kind="bar", figsize=(10, 5), color=["#CBD5E1", "#FBBF24", "#F97316", "#2563EB"])
plt.title("Annotation count distribution")
plt.xlabel("Event")
plt.ylabel("Number of clips")
plt.legend(title="Count out of 3")
plt.xticks(rotation=25, ha="right")
save_current_figure("annotation_count_distribution.png")
plt.show()

cooccurrence = pd.DataFrame(hard_targets, columns=EVENTS).corr().fillna(0)
plt.figure(figsize=(7, 6))
sns.heatmap(cooccurrence, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True)
plt.title("Majority-label event co-occurrence (phi correlation)")
save_current_figure("event_cooccurrence.png")
plt.show()

plt.figure(figsize=(8, 4.5))
sns.histplot(durations, bins=30, color="#2563EB")
plt.xlabel("Duration (seconds)")
plt.title("Audio duration distribution (no truncation)")
save_current_figure("audio_duration_distribution.png")
plt.show()

ambiguity_by_event = pd.Series(((counts_matrix == 1) | (counts_matrix == 2)).mean(axis=0), index=EVENTS)
ax = ambiguity_by_event.sort_values().plot(kind="barh", color="#F59E0B", figsize=(8, 4.5))
ax.set(xlabel="Fraction with count 1 or 2", ylabel="Event", title="B. Annotator disagreement proxy")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
save_current_figure("B_annotator_disagreement.png")
plt.show()

display(pd.DataFrame({"majority_prevalence": prevalence, "ambiguity_rate": ambiguity_by_event}))

## 5. Train, validation, and test splits

SHA-256 waveform fingerprints keep exact duplicates together across the 70/15/15 split. All variants reuse the same indices. Without verified speaker IDs, this remains a clip-level evaluation with possible speaker leakage. Test data is reserved for final evaluation.

In [ ]:
all_indices = np.arange(len(metadata))
groups = metadata["audio_sha256"].to_numpy()
if len(np.unique(groups)) < 3:
    raise RuntimeError("At least three unique clips are required to create train/validation/test splits.")

outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=RANDOM_SEED)
train_idx, temp_idx = next(outer.split(all_indices, groups=groups))
inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=RANDOM_SEED + 1)
val_rel, test_rel = next(inner.split(temp_idx, groups=groups[temp_idx]))
val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]

split_indices = {
    "train": np.sort(train_idx).tolist(),
    "validation": np.sort(val_idx).tolist(),
    "test": np.sort(test_idx).tolist(),
}
assert not (set(split_indices["train"]) & set(split_indices["validation"]))
assert not (set(split_indices["train"]) & set(split_indices["test"]))
assert not (set(split_indices["validation"]) & set(split_indices["test"]))
for left, right in ((train_idx, val_idx), (train_idx, test_idx), (val_idx, test_idx)):
    assert set(groups[left]).isdisjoint(set(groups[right]))

coverage_rows = []
for split_name, indices in split_indices.items():
    split_hard = hard_targets[indices]
    for event_i, event in enumerate(EVENTS):
        coverage_rows.append({
            "split": split_name,
            "n_clips": len(indices),
            "event": event,
            "positives": int(split_hard[:, event_i].sum()),
            "negatives": int(len(indices) - split_hard[:, event_i].sum()),
            "prevalence": float(split_hard[:, event_i].mean()),
        })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
print({name: len(idx) for name, idx in split_indices.items()})

## 6. WavLM feature extraction

WavLM Base+ is frozen and evaluated in batches. Attention-masked mean pooling excludes padded frames. A shared cache supplies identical embeddings to every model.

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
encoder = WavLMModel.from_pretrained(MODEL_NAME)
encoder.requires_grad_(False)
encoder.eval().to(DEVICE)

cache_key_payload = {
    "model": MODEL_NAME,
    "dataset_revision": RESOLVED_DATASET_REVISION,
    "sample_rate": SAMPLE_RATE,
    "fingerprints": metadata["audio_sha256"].tolist(),
}
cache_key = hashlib.sha256(json.dumps(cache_key_payload, sort_keys=True).encode()).hexdigest()[:16]
embedding_cache = OUTPUT_PATH / f"wavlm_embeddings_{cache_key}.npz"

def masked_mean_pool(hidden_states, input_attention_mask, model):
    if hasattr(model, "_get_feature_vector_attention_mask"):
        frame_mask = model._get_feature_vector_attention_mask(hidden_states.shape[1], input_attention_mask)
    else:
        input_lengths = input_attention_mask.sum(dim=-1)
        if hasattr(model, "_get_feat_extract_output_lengths"):
            output_lengths = model._get_feat_extract_output_lengths(input_lengths)
        else:
            output_lengths = torch.ceil(
                input_lengths.float() * hidden_states.shape[1] / input_attention_mask.shape[1]
            ).long()
        positions = torch.arange(hidden_states.shape[1], device=hidden_states.device)[None, :]
        frame_mask = positions < output_lengths[:, None]
    weights = frame_mask.unsqueeze(-1).to(hidden_states.dtype)
    return (hidden_states * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)

if embedding_cache.exists():
    cached = np.load(embedding_cache)
    embeddings = cached["embeddings"]
    if embeddings.shape[0] != len(waveforms):
        raise RuntimeError("Embedding cache length mismatch.")
    print("Loaded cached embeddings:", embedding_cache, embeddings.shape)
else:
    embedding_chunks = []
    autocast_enabled = DEVICE.type == "cuda"
    with torch.inference_mode():
        for start in tqdm(range(0, len(waveforms), RUN_BATCH_SIZE), desc="Extracting WavLM embeddings"):
            batch_audio = waveforms[start:start + RUN_BATCH_SIZE]
            inputs = feature_extractor(
                batch_audio,
                sampling_rate=SAMPLE_RATE,
                padding=True,
                truncation=False,
                return_attention_mask=True,
                return_tensors="pt",
            )
            input_values = inputs["input_values"].to(DEVICE)
            attention_mask = inputs.get("attention_mask", torch.ones_like(input_values, dtype=torch.long)).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=autocast_enabled):
                hidden = encoder(input_values=input_values, attention_mask=attention_mask).last_hidden_state
                pooled = masked_mean_pool(hidden, attention_mask, encoder)
            embedding_chunks.append(pooled.float().cpu().numpy())
    embeddings = np.concatenate(embedding_chunks).astype(np.float32)
    np.savez_compressed(embedding_cache, embeddings=embeddings)
    print("Saved shared embedding cache:", embedding_cache, embeddings.shape)

assert embeddings.shape == (len(waveforms), encoder.config.hidden_size)
encoder.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Models and objectives

All variants share the same architecture and training controls:

`WavLM embedding → Linear(256) → ReLU → Dropout → Linear(5)`

Model C adds `λ · mean((sigmoid(logit) − soft_target)²)` to soft-label BCE. Optional target-derived disagreement weighting is disabled. The objective models label ambiguity, not epistemic uncertainty.

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, x, hard_y, soft_y, indices):
        self.x = torch.from_numpy(x[indices]).float()
        self.hard_y = torch.from_numpy(hard_y[indices]).float()
        self.soft_y = torch.from_numpy(soft_y[indices]).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.hard_y[index], self.soft_y[index]


class StutterMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=5, dropout=DROPOUT):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.network(x)


datasets_by_split = {
    name: EmbeddingDataset(embeddings, hard_targets, soft_targets, indices)
    for name, indices in split_indices.items()
}

def make_loader(split_name, shuffle=False):
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    return DataLoader(
        datasets_by_split[split_name],
        batch_size=RUN_BATCH_SIZE,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )

def objective(logits, hard_y, soft_y, variant):
    if variant == "hard":
        return nn.functional.binary_cross_entropy_with_logits(logits, hard_y)
    element_bce = nn.functional.binary_cross_entropy_with_logits(logits, soft_y, reduction="none")
    if variant == "soft":
        return element_bce.mean()
    if variant == "calibrated":
        if USE_DISAGREEMENT_WEIGHTS:
            ambiguity = 4.0 * soft_y * (1.0 - soft_y)
            weights = 1.0 + DISAGREEMENT_WEIGHT_STRENGTH * ambiguity
            weights = weights / weights.mean().detach().clamp_min(1e-8)
            bce = (element_bce * weights).mean()
        else:
            bce = element_bce.mean()
        brier = torch.mean((torch.sigmoid(logits) - soft_y) ** 2)
        return bce + BRIER_LAMBDA * brier
    raise ValueError(f"Unknown variant: {variant}")

print(StutterMLP(embeddings.shape[1]))

## 8. Training

Each model uses AdamW and validation-only early stopping, then restores its best checkpoint. Class weighting is omitted to preserve probability interpretation. Test data is not used for model selection.

In [ ]:
def train_variant(variant, display_name):
    set_seed(RANDOM_SEED)
    model = StutterMLP(embeddings.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    train_loader = make_loader("train", shuffle=True)
    val_loader = make_loader("validation", shuffle=False)
    checkpoint_path = OUTPUT_PATH / f"{variant}_best.pt"
    history = {"train_loss": [], "validation_loss": []}
    best_val = float("inf")
    stale_epochs = 0

    for epoch in range(RUN_EPOCHS):
        model.train()
        running, seen = 0.0, 0
        progress = tqdm(train_loader, desc=f"{display_name} epoch {epoch + 1}/{RUN_EPOCHS}", leave=False)
        for x, hard_y, soft_y in progress:
            x, hard_y, soft_y = x.to(DEVICE), hard_y.to(DEVICE), soft_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = objective(logits, hard_y, soft_y, variant)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(x)
            seen += len(x)
            progress.set_postfix(loss=f"{running / seen:.4f}")
        train_loss = running / seen

        model.eval()
        val_running, val_seen = 0.0, 0
        with torch.inference_mode():
            for x, hard_y, soft_y in val_loader:
                x, hard_y, soft_y = x.to(DEVICE), hard_y.to(DEVICE), soft_y.to(DEVICE)
                loss = objective(model(x), hard_y, soft_y, variant)
                val_running += loss.item() * len(x)
                val_seen += len(x)
        val_loss = val_running / val_seen
        history["train_loss"].append(train_loss)
        history["validation_loss"].append(val_loss)
        print(f"{display_name}: epoch {epoch + 1:02d} | train {train_loss:.5f} | val {val_loss:.5f}")

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            stale_epochs = 0
            cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save({
                "model_state_dict": cpu_state,
                "variant": variant,
                "input_dim": embeddings.shape[1],
                "events": EVENTS,
                "best_validation_loss": best_val,
                "seed": RANDOM_SEED,
            }, checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping {display_name}; best validation objective = {best_val:.5f}")
                break

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval().cpu()
    return model, history, checkpoint_path


variant_names = {
    "hard": "A — hard-label BCE",
    "soft": "B — soft-label BCE",
    "calibrated": "C — soft BCE + Brier penalty",
}
trained_models, histories, checkpoint_paths = {}, {}, {}
for variant, display_name in variant_names.items():
    model, history, path = train_variant(variant, display_name)
    trained_models[variant] = model
    histories[variant] = history
    checkpoint_paths[variant] = str(path)

In [ ]:
plt.figure(figsize=(10, 5.5))
colors = {"hard": "#2563EB", "soft": "#16A34A", "calibrated": "#DC2626"}
for variant, history in histories.items():
    epochs_axis = np.arange(1, len(history["train_loss"]) + 1)
    plt.plot(epochs_axis, history["train_loss"], "--", color=colors[variant], alpha=0.65,
             label=f"{variant_names[variant]} train")
    plt.plot(epochs_axis, history["validation_loss"], "-", color=colors[variant],
             label=f"{variant_names[variant]} validation")
plt.xlabel("Epoch")
plt.ylabel("Variant training objective")
plt.title("C. Training and validation loss curves")
plt.legend(fontsize=8, ncol=2)
save_current_figure("C_training_validation_curves.png")
plt.show()

## 9. Evaluation

Models are evaluated on the same test split at threshold 0.5. Optional thresholds are tuned on validation data only. Binary-label and annotator-proportion calibration are reported separately. Selective risk retains clips in order of increasing predictive entropy.

In [ ]:
def predict_probabilities(model, indices):
    model = model.to(DEVICE).eval()
    x = torch.from_numpy(embeddings[indices]).float()
    loader = DataLoader(x, batch_size=max(RUN_BATCH_SIZE, 32), shuffle=False)
    chunks = []
    with torch.inference_mode():
        for batch in loader:
            chunks.append(torch.sigmoid(model(batch.to(DEVICE))).cpu().numpy())
    model.cpu()
    return np.concatenate(chunks)

def binary_entropy(probabilities):
    p = np.clip(probabilities, 1e-7, 1 - 1e-7)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def expected_calibration_error(probabilities, targets, n_bins=10):
    p = np.asarray(probabilities).ravel()
    y = np.asarray(targets).ravel()
    edges = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.minimum(np.digitize(p, edges[1:-1], right=False), n_bins - 1)
    ece = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if mask.any():
            ece += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return float(ece)

def safe_rank_metric(function, y_true, y_score):
    if np.unique(y_true).size < 2:
        return np.nan
    try:
        return float(function(y_true, y_score))
    except ValueError:
        return np.nan

def tune_thresholds(validation_hard, validation_probabilities):
    thresholds = []
    grid = np.linspace(0.05, 0.95, 19)
    for event_i in range(len(EVENTS)):
        scores = [
            f1_score(validation_hard[:, event_i], validation_probabilities[:, event_i] >= threshold,
                     zero_division=0)
            for threshold in grid
        ]
        thresholds.append(float(grid[int(np.argmax(scores))]))
    return np.asarray(thresholds)

def evaluate_variant(variant, test_probabilities, thresholds):
    y_hard = hard_targets[test_idx]
    y_soft = soft_targets[test_idx]
    y_pred = (test_probabilities >= thresholds).astype(int)
    per_event_auroc = [safe_rank_metric(roc_auc_score, y_hard[:, j], test_probabilities[:, j]) for j in range(len(EVENTS))]
    per_event_ap = [safe_rank_metric(average_precision_score, y_hard[:, j], test_probabilities[:, j]) for j in range(len(EVENTS))]
    precision, recall, per_f1, support = precision_recall_fscore_support(
        y_hard, y_pred, average=None, zero_division=0
    )
    entropy_per_clip = binary_entropy(test_probabilities).mean(axis=1)
    ambiguity_per_clip = ((counts_matrix[test_idx] == 1) | (counts_matrix[test_idx] == 2)).mean(axis=1)
    correlation = spearmanr(entropy_per_clip, ambiguity_per_clip).statistic
    sample_error = np.not_equal(y_pred, y_hard).mean(axis=1)
    low_agreement = ambiguity_per_clip > 0
    metrics = {
        "model": variant,
        "macro_f1": float(f1_score(y_hard, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_hard, y_pred, average="micro", zero_division=0)),
        "macro_auroc": float(np.nanmean(per_event_auroc)) if np.isfinite(per_event_auroc).any() else np.nan,
        "macro_average_precision": float(np.nanmean(per_event_ap)) if np.isfinite(per_event_ap).any() else np.nan,
        "hard_brier": float(np.mean((test_probabilities - y_hard) ** 2)),
        "soft_brier": float(np.mean((test_probabilities - y_soft) ** 2)),
        "hard_ece": expected_calibration_error(test_probabilities, y_hard),
        "soft_ece": expected_calibration_error(test_probabilities, y_soft),
        "mean_confidence": float(np.maximum(test_probabilities, 1 - test_probabilities).mean()),
        "mean_prediction_entropy": float(entropy_per_clip.mean()),
        "uncertainty_ambiguity_spearman": float(correlation) if np.isfinite(correlation) else np.nan,
        "high_agreement_hamming_error": float(sample_error[~low_agreement].mean()) if (~low_agreement).any() else np.nan,
        "low_agreement_hamming_error": float(sample_error[low_agreement].mean()) if low_agreement.any() else np.nan,
    }
    event_rows = []
    for j, event in enumerate(EVENTS):
        event_rows.append({
            "model": variant, "event": event,
            "precision": float(precision[j]), "recall": float(recall[j]), "f1": float(per_f1[j]),
            "support": int(support[j]), "auroc": per_event_auroc[j], "average_precision": per_event_ap[j],
        })
    return metrics, event_rows, entropy_per_clip, sample_error


val_probabilities = {v: predict_probabilities(m, val_idx) for v, m in trained_models.items()}
test_probabilities = {v: predict_probabilities(m, test_idx) for v, m in trained_models.items()}
thresholds_by_variant = {}
summary_rows, per_event_rows = [], []
uncertainty_by_variant, errors_by_variant = {}, {}
for variant in variant_names:
    thresholds = (tune_thresholds(hard_targets[val_idx], val_probabilities[variant])
                  if TUNE_THRESHOLDS_ON_VALIDATION else np.full(len(EVENTS), 0.5))
    thresholds_by_variant[variant] = thresholds
    row, event_rows, uncertainty, errors = evaluate_variant(variant, test_probabilities[variant], thresholds)
    row["threshold_policy"] = "validation_tuned" if TUNE_THRESHOLDS_ON_VALIDATION else "fixed_0.5"
    summary_rows.append(row)
    per_event_rows.extend(event_rows)
    uncertainty_by_variant[variant] = uncertainty
    errors_by_variant[variant] = errors

comparison_df = pd.DataFrame(summary_rows).set_index("model")
per_class_df = pd.DataFrame(per_event_rows)
display(comparison_df)
display(per_class_df)

## 10. Visualizations and uncertainty

The following figures compare classification, calibration, selective risk, disagreement alignment, and multi-label errors. Lower Brier, ECE, and risk are better; higher F1, AUROC, and average precision are better.

In [ ]:
plot_f1 = per_class_df.pivot(index="event", columns="model", values="f1").reindex(EVENTS)
plot_f1.plot(kind="bar", figsize=(10, 5), color=[colors[c] for c in plot_f1.columns])
plt.ylim(0, 1)
plt.ylabel("F1 at selected threshold policy")
plt.xlabel("Event")
plt.title("D. Per-class F1 comparison")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Model")
save_current_figure("D_per_class_f1.png")
plt.show()

In [ ]:
def reliability_points(probabilities, targets, n_bins=10):
    p, y = probabilities.ravel(), targets.ravel()
    edges = np.linspace(0, 1, n_bins + 1)
    ids = np.minimum(np.digitize(p, edges[1:-1]), n_bins - 1)
    x, observed, sizes = [], [], []
    for b in range(n_bins):
        mask = ids == b
        if mask.any():
            x.append(float(p[mask].mean()))
            observed.append(float(y[mask].mean()))
            sizes.append(int(mask.sum()))
    return np.asarray(x), np.asarray(observed), np.asarray(sizes)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, target, target_name in zip(
    axes,
    [hard_targets[test_idx], soft_targets[test_idx]],
    ["Majority-vote binary target", "Annotator-proportion target"],
):
    ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect calibration")
    for variant in variant_names:
        x, observed, _ = reliability_points(test_probabilities[variant], target)
        ax.plot(x, observed, marker="o", color=colors[variant], label=variant)
    ax.set(title=target_name, xlabel="Mean predicted probability", ylabel="Observed target mean", xlim=(0, 1), ylim=(0, 1))
axes[1].legend(fontsize=8)
fig.suptitle("E. Reliability diagrams: binary and annotator-proportion calibration")
fig.tight_layout()
save_current_figure("E_reliability_diagrams.png")
plt.show()

In [ ]:
coverages = np.linspace(0.1, 1.0, 10)
risk_rows = []
for variant in variant_names:
    order = np.argsort(uncertainty_by_variant[variant])
    for coverage in coverages:
        retained = order[:max(1, int(math.ceil(coverage * len(order))))]
        risk_rows.append({
            "model": variant,
            "coverage": float(coverage),
            "hamming_risk": float(errors_by_variant[variant][retained].mean()),
        })
risk_coverage_df = pd.DataFrame(risk_rows)
plt.figure(figsize=(8, 5))
for variant in variant_names:
    subset = risk_coverage_df[risk_coverage_df.model == variant]
    plt.plot(subset.coverage, subset.hamming_risk, marker="o", color=colors[variant], label=variant)
plt.xlabel("Coverage (least-uncertain clips retained)")
plt.ylabel("Hamming risk")
plt.title("F. Selective risk–coverage")
plt.legend()
save_current_figure("F_risk_coverage.png")
plt.show()
display(risk_coverage_df)

In [ ]:
ambiguity_test = ((counts_matrix[test_idx] == 1) | (counts_matrix[test_idx] == 2)).mean(axis=1)
ambiguity_rows = []
for variant in variant_names:
    for ambiguity_value in np.sort(np.unique(ambiguity_test)):
        mask = ambiguity_test == ambiguity_value
        ambiguity_rows.append({
            "model": variant,
            "ambiguous_target_fraction": float(ambiguity_value),
            "mean_prediction_entropy": float(uncertainty_by_variant[variant][mask].mean()),
            "n_clips": int(mask.sum()),
        })
ambiguity_uncertainty_df = pd.DataFrame(ambiguity_rows)
plt.figure(figsize=(9, 5))
for variant in variant_names:
    subset = ambiguity_uncertainty_df[ambiguity_uncertainty_df.model == variant]
    plt.plot(subset.ambiguous_target_fraction, subset.mean_prediction_entropy,
             marker="o", color=colors[variant], label=variant)
plt.xlabel("Fraction of event targets with count 1 or 2")
plt.ylabel("Mean binary prediction entropy (bits)")
plt.title("G. Human disagreement proxy vs model uncertainty")
plt.legend()
save_current_figure("G_disagreement_vs_uncertainty.png")
plt.show()
display(ambiguity_uncertainty_df)

In [ ]:
error_panels = []
y_hard_test = hard_targets[test_idx].astype(int)
for variant in variant_names:
    pred = (test_probabilities[variant] >= thresholds_by_variant[variant]).astype(int)
    rows = []
    for j, event in enumerate(EVENTS):
        yt, yp = y_hard_test[:, j], pred[:, j]
        rows.append({
            "event": event,
            "false positive rate": float(((yt == 0) & (yp == 1)).mean()),
            "false negative rate": float(((yt == 1) & (yp == 0)).mean()),
        })
    error_panels.append(pd.DataFrame(rows).set_index("event"))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, variant, panel in zip(axes, variant_names, error_panels):
    sns.heatmap(panel, annot=True, fmt=".3f", cmap="magma", vmin=0,
                vmax=max(0.01, max(p.to_numpy().max() for p in error_panels)), ax=ax, cbar=ax is axes[-1])
    ax.set_title(variant)
    ax.set_xlabel("Test-set fraction")
fig.suptitle("H. Multi-label false-positive and false-negative rates")
fig.tight_layout()
save_current_figure("H_multilabel_error_analysis.png")
plt.show()

agreement_error_df = comparison_df[["high_agreement_hamming_error", "low_agreement_hamming_error"]]
display(agreement_error_df)

## 11. Results summary

The summary is generated from measured test results. Rankings describe this split only and do not imply statistical significance.

In [ ]:
def finite_best(series, minimize=False):
    valid = series.dropna()
    if valid.empty:
        return "undefined", np.nan
    name = valid.idxmin() if minimize else valid.idxmax()
    return name, float(valid.loc[name])

best_f1_model, best_f1 = finite_best(comparison_df["macro_f1"])
best_soft_brier_model, best_soft_brier = finite_best(comparison_df["soft_brier"], minimize=True)
best_hard_brier_model, best_hard_brier = finite_best(comparison_df["hard_brier"], minimize=True)
soft_delta = comparison_df.loc["soft", "soft_brier"] - comparison_df.loc["hard", "soft_brier"]
cal_delta = comparison_df.loc["calibrated", "soft_brier"] - comparison_df.loc["hard", "soft_brier"]

summary_text = f'''# StutterSense experiment summary

- The fixed held-out clip-level test split contained {len(test_idx)} clips; it was duplicate-aware but not verified speaker-disjoint.
- Highest macro F1: **{best_f1_model}** ({best_f1:.4f}).
- Lowest annotator-proportion Brier score: **{best_soft_brier_model}** ({best_soft_brier:.4f}).
- Lowest majority-label Brier score: **{best_hard_brier_model}** ({best_hard_brier:.4f}).
- Relative to the hard-label baseline, the soft-label model changed soft-target Brier by {soft_delta:+.4f}; the calibrated model changed it by {cal_delta:+.4f}. Negative is better.

These measurements describe this run only. They do not demonstrate statistical significance, speaker-independent generalization, epistemic uncertainty, or clinical validity.
'''
display(Markdown(summary_text))

limitations = '''
## Limitations and threats to validity

- The hosted default subset is not assumed to equal the complete original SEP-28k corpus.
- No verified speaker or episode IDs means potential speaker/content leakage across the clip-level split.
- Aggregate counts reveal ambiguity but not annotator identities, dependencies, or individual reliability.
- A 2-of-3 majority threshold discards information and can be unstable near the boundary.
- Frozen embeddings and one MLP family limit the scope of architectural conclusions.
- One random split and seed do not quantify sampling or optimization variability.
- Predictive entropy mixes data ambiguity, model fit, and class prevalence; it is not a pure measure of epistemic uncertainty.
- Class imbalance can make thresholded F1 unstable, especially when a test event has few positives.
- These outputs are research predictions, not clinical diagnoses.

## Suggested follow-up experiments

- Obtain verified speaker/episode IDs and perform grouped speaker-disjoint cross-validation.
- Repeat across seeds and bootstrap clips or speakers for confidence intervals.
- Compare frozen features with carefully regularized end-to-end fine-tuning.
- Evaluate deep ensembles or Bayesian approximations for epistemic uncertainty.
- Model individual annotators when rater-level labels become available.
- Pre-register primary calibration metrics and test on an external corpus.
'''
display(Markdown(limitations))

## 12. Interactive demo

Select a loaded clip or upload audio to inspect its waveform, event probabilities, and mean binary entropy. Uploaded audio is converted to mono at 16 kHz without truncation.

**Research use only; predictions are not clinical diagnoses.**

In [ ]:
import ipywidgets as widgets

def embedding_for_waveform(waveform):
    encoder.to(DEVICE).eval()
    inputs = feature_extractor(
        [waveform], sampling_rate=SAMPLE_RATE, padding=True, truncation=False,
        return_attention_mask=True, return_tensors="pt",
    )
    values = inputs["input_values"].to(DEVICE)
    mask = inputs.get("attention_mask", torch.ones_like(values, dtype=torch.long)).to(DEVICE)
    with torch.inference_mode():
        hidden = encoder(input_values=values, attention_mask=mask).last_hidden_state
        pooled = masked_mean_pool(hidden, mask, encoder)
    encoder.cpu()
    return pooled.float().cpu().numpy()

def predict_embedding(single_embedding):
    output = {}
    tensor = torch.from_numpy(single_embedding).float()
    for variant, model in trained_models.items():
        model.eval()
        with torch.inference_mode():
            output[variant] = torch.sigmoid(model(tensor)).numpy()[0]
    return output

def render_demo(waveform, title, true_counts=None, precomputed_embedding=None):
    clear_output(wait=True)
    display(Markdown(f"### {title}"))
    display(IPythonAudio(waveform, rate=SAMPLE_RATE))
    fig, ax = plt.subplots(figsize=(10, 2.5))
    time_axis = np.arange(len(waveform)) / SAMPLE_RATE
    ax.plot(time_axis, waveform, linewidth=0.7, color="#2563EB")
    ax.set(xlabel="Time (s)", ylabel="Amplitude", title="Waveform")
    plt.show()
    if true_counts is not None:
        display(pd.DataFrame({"event": EVENTS, "annotator_count": true_counts, "soft_target": np.asarray(true_counts) / 3}))
    emb = precomputed_embedding if precomputed_embedding is not None else embedding_for_waveform(waveform)
    probabilities = predict_embedding(emb)
    table = pd.DataFrame(probabilities, index=EVENTS)
    entropy = {name: float(binary_entropy(prob).mean()) for name, prob in probabilities.items()}
    display(table.style.format("{:.3f}"))
    display(pd.DataFrame({"model": list(entropy), "mean_binary_entropy_bits": list(entropy.values())}))
    display(Markdown("*Experimental research predictions only; not a clinical diagnosis.*"))

clip_selector = widgets.Dropdown(
    options=[(f"Clip {i}: {metadata.loc[i, 'duration_seconds']:.2f}s", i) for i in range(len(waveforms))],
    value=0,
    description="Clip:",
    layout=widgets.Layout(width="55%"),
)
clip_output = widgets.Output()

def update_clip(change=None):
    i = int(clip_selector.value)
    with clip_output:
        render_demo(
            waveforms[i], f"Loaded clip {i}",
            true_counts=counts_matrix[i].astype(int),
            precomputed_embedding=embeddings[i:i + 1],
        )

clip_selector.observe(update_clip, names="value")
display(clip_selector, clip_output)
update_clip()

In [ ]:
upload_widget = widgets.FileUpload(accept=".wav,audio/*", multiple=False, description="Upload audio")
upload_output = widgets.Output()

def uploaded_bytes(widget_value):
    if isinstance(widget_value, dict):
        item = next(iter(widget_value.values()))
    else:
        item = widget_value[0]
    content = item["content"]
    return content.tobytes() if hasattr(content, "tobytes") else bytes(content)

def handle_upload(change):
    if not upload_widget.value:
        return
    with upload_output:
        try:
            payload = uploaded_bytes(upload_widget.value)
            waveform, original_rate = decode_audio_value({"bytes": payload, "path": None})
            render_demo(waveform, f"Uploaded audio (source rate {original_rate} Hz)")
        except Exception as exc:
            clear_output(wait=True)
            print("Could not decode uploaded audio:", repr(exc))

upload_widget.observe(handle_upload, names="value")
display(Markdown("### Optional arbitrary-audio inference"), upload_widget, upload_output)

## 13. Export

Export configuration, provenance, splits, metrics, histories, figures, checkpoints, and a portable ZIP archive.

In [ ]:
config_export = {
    "dataset_id": DATASET_ID, "dataset_config": DATASET_CONFIG, "dataset_split": "train",
    "dataset_revision": RESOLVED_DATASET_REVISION,
    "max_samples": MAX_SAMPLES, "run_max_samples": RUN_MAX_SAMPLES,
    "batch_size": BATCH_SIZE, "run_batch_size": RUN_BATCH_SIZE,
    "epochs": EPOCHS, "run_epochs": RUN_EPOCHS,
    "learning_rate": LEARNING_RATE, "random_seed": RANDOM_SEED,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE, "dropout": DROPOUT,
    "sample_rate": SAMPLE_RATE, "model_name": MODEL_NAME,
    "brier_lambda": BRIER_LAMBDA,
    "use_disagreement_weights": USE_DISAGREEMENT_WEIGHTS,
    "disagreement_weight_strength": DISAGREEMENT_WEIGHT_STRENGTH,
    "threshold_policy": "validation_tuned" if TUNE_THRESHOLDS_ON_VALIDATION else "fixed_0.5",
    "fast_smoke_test": FAST_SMOKE_TEST, "cpu_smoke_test": CPU_SMOKE_TEST,
    "output_dir": str(OUTPUT_PATH.resolve()),
}
provenance = {
    "dataset_id": DATASET_ID,
    "configuration": DATASET_CONFIG,
    "split": "train",
    "resolved_revision": RESOLVED_DATASET_REVISION,
    "streaming": True,
    "materialized_successful_clips": len(metadata),
    "attempted_stream_examples": attempted,
    "decode_failures": len(decode_failures),
    "verified_columns": available_columns,
    "identifier_like_columns": identifier_candidates,
    "audio_feature_repr": repr(raw_stream.features.get("audio")),
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation_scope": "duplicate-aware clip-level exploratory split; not verified speaker-disjoint",
    "package_versions": {
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "datasets": datasets.__version__,
        "numpy": np.__version__, "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__, "pandas": pd.__version__,
    },
}

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

(OUTPUT_PATH / "config.json").write_text(json.dumps(json_safe(config_export), indent=2))
(OUTPUT_PATH / "dataset_provenance.json").write_text(json.dumps(json_safe(provenance), indent=2))
(OUTPUT_PATH / "split_indices.json").write_text(json.dumps(json_safe(split_indices), indent=2))
(OUTPUT_PATH / "thresholds.json").write_text(json.dumps(json_safe(thresholds_by_variant), indent=2))
(OUTPUT_PATH / "training_histories.json").write_text(json.dumps(json_safe(histories), indent=2))
(OUTPUT_PATH / "evaluation_metrics.json").write_text(
    json.dumps(json_safe({"summary": summary_rows, "per_class": per_event_rows}), indent=2)
)
comparison_df.to_csv(OUTPUT_PATH / "comparison_table.csv")
per_class_df.to_csv(OUTPUT_PATH / "per_class_metrics.csv", index=False)
risk_coverage_df.to_csv(OUTPUT_PATH / "risk_coverage.csv", index=False)
ambiguity_uncertainty_df.to_csv(OUTPUT_PATH / "ambiguity_uncertainty.csv", index=False)
metadata.drop(columns=[], errors="ignore").to_csv(OUTPUT_PATH / "materialized_clip_metadata.csv", index=False)
(OUTPUT_PATH / "experiment_summary.md").write_text(summary_text + limitations)

zip_base = str(OUTPUT_PATH.parent / "stuttersense_outputs")
zip_path = Path(shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_PATH))
print(f"Exported {len(list(OUTPUT_PATH.rglob('*')))} files to {OUTPUT_PATH}")
print("ZIP archive:", zip_path)

if SHOW_OUTPUT_ZIP_LINK:
    display(FileLink(str(zip_path), result_html_prefix="Download the output archive: "))